# EDA del dataset (T4.4)

Versión mínima sobre `data/noticias_politica.parquet` (fase 4): volumen por medio y mes, largo de las notas, bajada, notas en vivo y `content_code` de La Nación. Alimenta las decisiones de la fase 5: unidad de análisis y granularidad de los agregados.

Infobae y La Nación son una muestra de 30 notas por mes (hash de la URL, catálogo de Infobae con los sitemaps de los días 5 y 20); MisionesOnline está completo. Período: 10/12/2023 a 30/09/2026 23:59 ART. Corre sin red.

In [1]:
import json
from pathlib import Path

import pandas as pd

DATA = Path("../data")
pd.set_option("display.max_colwidth", 80)

df = pd.read_parquet(DATA / "noticias_politica.parquet")
df["mes"] = df["fecha_publicacion"].dt.strftime("%Y-%m")
df["palabras"] = df["cuerpo"].str.split().str.len().fillna(0).astype(int)
print(len(df), "notas,", df["fecha_publicacion"].min(), "a", df["fecha_publicacion"].max())
df["medio"].value_counts()

9261 notas, 2023-12-10 09:03:09-03:00 a 2026-09-30 23:37:19-03:00


medio
misionesonline    7245
lanacion          1020
infobae            996
Name: count, dtype: int64

## 1. Volumen por medio y mes

In [2]:
por_mes = df.pivot_table(index="mes", columns="medio", values="id", aggfunc="count")
print("Meses por medio:", por_mes.notna().sum().to_dict())
por_mes.describe().round(0)

Meses por medio: {'infobae': 34, 'lanacion': 34, 'misionesonline': 34}


medio,infobae,lanacion,misionesonline
count,34.0,34.0,34.0
mean,29.0,30.0,213.0
std,2.0,0.0,76.0
min,21.0,30.0,90.0
25%,30.0,30.0,161.0
50%,30.0,30.0,190.0
75%,30.0,30.0,288.0
max,30.0,30.0,355.0


In [3]:
semanas = df[df["medio"] == "misionesonline"].groupby(df["fecha_publicacion"].dt.strftime("%G-W%V")).size()
print("MisionesOnline, notas por semana:", semanas.describe().round(0).to_dict())
por_mes.T

MisionesOnline, notas por semana: {'count': 148.0, 'mean': 49.0, 'std': 21.0, 'min': 10.0, '25%': 35.0, '50%': 45.0, '75%': 61.0, 'max': 122.0}


mes,2023-12,2024-01,2024-02,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,...,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08,2026-09
medio,,,,,,,,,,,,,,,,,,,,,
infobae,30,30,30,30,30,30,30,30,30,30,...,30,30,30,30,24,30,30,23,30,30
lanacion,30,30,30,30,30,30,30,30,30,30,...,30,30,30,30,30,30,30,30,30,30
misionesonline,207,134,143,171,150,203,166,122,143,195,...,178,90,177,317,322,340,335,291,316,319


## 2. Largo de las notas

In [4]:
largo = df.assign(
    palabras_titulo=df["titulo"].str.split().str.len(),
    palabras_bajada=df["bajada"].str.split().str.len(),
    cuerpo_vacio=df["palabras"].eq(0),
)
largo.groupby("medio").agg(
    cuerpo_mediana=("palabras", "median"),
    cuerpo_p90=("palabras", lambda s: s.quantile(0.9)),
    cuerpo_vacio=("cuerpo_vacio", "sum"),
    titulo_mediana=("palabras_titulo", "median"),
    bajada_mediana=("palabras_bajada", "median"),
)

,cuerpo_mediana,cuerpo_p90,cuerpo_vacio,titulo_mediana,bajada_mediana
medio,,,,,
infobae,794.0,1365.5,0,18.0,34.0
lanacion,751.5,1482.1,9,17.0,28.0
misionesonline,478.0,962.6,4,18.0,39.0


## 3. Bajada y notas reescritas

In [5]:
df.assign(con_bajada=df["bajada"].ne(""), con_fuente=df["fuente_externa"].ne("")).groupby("medio")[["con_bajada", "con_fuente"]].mean().round(3)

,con_bajada,con_fuente
medio,,
infobae,1.000,0.000
lanacion,0.993,0.000
misionesonline,0.694,0.127


En la fase 3, la corrida de prueba de octubre de 2026 dio menos bajada (47 %) y menos "Fuente: X" (5 %) en MisionesOnline que la muestra de la fase 2. Por trimestre:

In [6]:
mo = df[df["medio"] == "misionesonline"]
trimestre = mo["fecha_publicacion"].dt.year.astype(str) + "-T" + mo["fecha_publicacion"].dt.quarter.astype(str)
mo.assign(trimestre=trimestre, con_bajada=mo["bajada"].ne(""), con_fuente=mo["fuente_externa"].ne("")).groupby("trimestre")[["con_bajada", "con_fuente"]].mean().round(3)

,con_bajada,con_fuente
trimestre,,
2023-T4,0.986,0.155
2024-T1,0.978,0.272
2024-T2,0.987,0.395
2024-T3,0.900,0.204
2024-T4,0.682,0.205
2025-T1,0.530,0.172
2025-T2,0.585,0.037
2025-T3,0.539,0.057
2025-T4,0.541,0.119


## 4. Notas en vivo

In [7]:
df.groupby("medio")["es_en_vivo"].agg(["sum", "mean"]).round(3)

,sum,mean
medio,,
infobae,14,0.014
lanacion,31,0.030
misionesonline,0,0.000


In [8]:
raw_ln = pd.concat(pd.read_parquet(p) for p in sorted((DATA / "raw_lanacion").glob("*.parquet"))).dropna(subset=["payload"])
stories = {}
for payload in raw_ln["payload"]:
    story = json.loads(payload)
    stories["https://www.lanacion.com.ar" + story["canonical_url"]] = story
ln = df[df["medio"] == "lanacion"].copy()
ln["subtype"] = ln["url"].map(lambda u: stories.get(u, {}).get("subtype"))
ln["content_code"] = ln["url"].map(lambda u: (stories.get(u, {}).get("content_restrictions") or {}).get("content_code"))
vivo = ln[ln["es_en_vivo"]]
print("La Nación en vivo: palabras", vivo["palabras"].describe().round(0).to_dict())
print("URL con 'en-vivo':", vivo["url"].str.contains("en-vivo").sum(), "de", len(vivo))
vivo[["subtype", "palabras", "titulo"]].sort_values("palabras")

La Nación en vivo: palabras {'count': 31.0, 'mean': 5431.0, 'std': 2060.0, 'min': 445.0, '25%': 4176.0, '50%': 5914.0, '75%': 6932.0, 'max': 8668.0}
URL con 'en-vivo': 31 de 31


,subtype,palabras,titulo
1156,1,445,Cynthia García lloró en vivo en C5N por las personas en situación de calle
1252,1,698,Cómo seguir a Lijo en vivo en el Senado: defiende su candidatura a la Corte ...
1952,11,2551,Georgieva señaló que la Argentina está en una posición “sólida” y agregó sob...
1248,6,2908,"Javier Milei, en vivo: las últimas medidas del Gobierno"
1914,11,3348,El comunicado de Pro sobre la renuncia de Adorni: “Era lo que correspondía”
1697,11,3353,"Javier Milei, en vivo: las últimas medidas del Gobierno"
1807,11,3816,Monteoliva le respondió a Javier Alonso: “Uno tiene que reconocer un problem...
1419,1,4022,La intendenta de Moreno cruzó a Lilia Lemoine por decir que está de vacacion...
1817,11,4329,Sturzenegger celebró que en 2025 la Argentina haya sido el país con la mayor...
1998,11,4548,La oposición presentó un proyecto en Diputados que declara la Emergencia en ...


## 5. Paywall de La Nación (`content_code`)

In [9]:
ln.groupby("content_code", dropna=False)["palabras"].describe().round(0)

,count,mean,std,min,25%,50%,75%,max
content_code,,,,,,,,
abierta,44.0,1365.0,1834.0,208.0,522.0,604.0,794.0,8668.0
comun,975.0,1012.0,1613.0,0.0,586.0,759.0,1010.0,42338.0
NaN,1.0,6613.0,NaN,6613.0,6613.0,6613.0,6613.0,6613.0


## Conclusiones para la fase 5

- **Granularidad.** Infobae y La Nación tienen 30 notas por mes (Infobae baja a 21–24 en algunos meses, como 2026-04 y 2026-07): los agregados de tono de los medios nacionales van por mes. MisionesOnline (mediana 45 notas por semana) admite semanas, pero para comparar los tres medios se usa el mes.
- **Volumen de MisionesOnline.** Pasa de ~170 notas por mes a ~320 desde marzo de 2026. No cambia el tono medio, pero sí cuánto pesa cada período si se agregan las notas sin ponderar.
- **Unidad de análisis.** Título + bajada: la bajada está en el 100 % de Infobae y el 99 % de La Nación (mediana de 28–34 palabras). En MisionesOnline falta en el 31 % (la regla la deja vacía cuando es el comienzo del cuerpo); ahí se usa título + primer párrafo.
- **Notas en vivo.** La Nación sí tiene (3 %): sobre todo la serie diaria "Javier Milei, en vivo: las últimas medidas del Gobierno", con una mediana de ~5.900 palabras. La regla por URL acierta en 29 de 31; los 2 falsos positivos son notas cortas ("lloró en vivo en C5N", "cómo seguir a Lijo en vivo"). Quedan fuera de los agregados, como estaba decidido; el error es menor.
- **Paywall.** En la muestra solo aparecen `comun` (975) y `abierta` (44), y `comun` no tiene cuerpos más cortos que `abierta` (mediana 759 contra 604 palabras): no hay señales de truncación, pero tampoco notas pagas para verificarla. Queda como limitación del informe.
- **MisionesOnline cambia en el tiempo.** La bajada propia baja de ~98 % (hasta 2024-T2) a ~55 % (desde 2025), y "Fuente: X" de 20–40 % (2024) a 3–12 % (desde 2025-T2). La comparación "con y sin `fuente_externa`" tiene que leerse con eso en mente.